# Importing, Inspecting, and Cleaning Data

Data files can come from many sources. They may be generated by experimental software, collected through online surveys, or entered manually into spreadsheets.

When learning R, we often use clean, well-structured example datasets. In practice, however, real-world data is frequently messy, inconsistent, or not immediately ready for analysis.

As a result, three tasks will be central to your workflow:

1. **Importing data** into R
2. **Inspecting data** to understand its structure and quality
3. **Cleaning data** so it is ready for reliable analysis

These are essential skills you will use regularly in nearly every data-analysis project.


## Importing data

Before analysing data, you need to import it into R. Data files come in many formats but one of the most common formats is the **CSV** file.

CSV stands for **comma-separated values**. If your data are stored in Microsoft Excel, you can export a spreadsheet as a CSV file by choosing **Save As** and selecting the CSV format.

This section focuses on CSV files, although R provides many functions and packages for working with other file types.

### Importing a CSV File from the Web

Your data may be stored on your computer or hosted online. The R function `read.csv()` can import data from either location.

```r
ICARData <- read.csv(
  "https://www.repository.cam.ac.uk/bitstreams/28056594-fd78-4847-ad94-dc6d5bda17de/download"
)
```

In this example:

- `read.csv()` downloads and imports the CSV file from the specified URL.
- `<-` assigns the imported data to a new object named `ICARData`.
- The data contain cognitive-ability scores measured using the International Cognitive Ability Resource (ICAR).

### Importing a CSV File from Your Computer

To import a file stored locally, replace the web URL with the file path or filename.

```r
ICARData <- read.csv("icardata.csv")
```

In RStudio, if the CSV file appears in the **Files** pane for your project, you can usually import it using only its filename.

### Useful `read.csv()` Arguments

| Argument | Purpose | Example |
|---|---|---|
| `file` | Specifies the file path or URL to import. | `file = "icardata.csv"` |
| `header` | Indicates whether the first row contains column names. The default is `TRUE`. | `header = FALSE` |
| `na.strings` | Specifies values that should be treated as missing data (`NA`). | `na.strings = c("nodata", "-999")` |

For example, if missing values were recorded as `"nodata"` or `-999`, convert them to `NA` during import:

```r
ICARData <- read.csv(
  "icardata.csv",
  na.strings = c("nodata", "-999")
)
```

Using `NA` for missing values is important because many R functions recognise it as the standard missing-data marker.




## Inspecting Data

Once your data have been imported, the next step is to inspect them to understand their structure and contents.

A useful function for this is `str()`, which displays the structure of an object:

```r
str(ICARData)
```

```text
'data.frame':  6208 obs. of  6 variables:
 $ Pcp     : chr  "02ackyqdw" "02ackyqdw" "02ackyqdw" "02ackyqdw" ...
 $ Item    : int  1 2 3 4 5 6 7 8 9 10 ...
 $ Response: int  4 4 4 6 6 3 4 1 5 2 ...
 $ Icartype: chr  "VR" "VR" "VR" "VR" ...
 $ Answer  : int  4 4 4 6 6 3 4 4 5 2 ...
 $ Correct : logi  TRUE TRUE TRUE TRUE TRUE TRUE ...
```

The output shows that the dataset has **6,208 rows** (observations) and **6 columns** (variables).

It also shows the data type assigned to each variable:

- `Pcp` and `Icartype` are character variables (`chr`).
- `Item`, `Response`, and `Answer` are integer variables (`int`).
- `Correct` is a logical variable (`logi`), containing `TRUE` or `FALSE` values.

R infers these data types from the contents of each column, but its choices may not always match how you intend to use the data. It is therefore important to check the results of an import before beginning an analysis.

This dataset was prepared for publication, so it does not contain obvious import problems. However, `Icartype` has been imported as a character variable even though it would likely be more appropriate to store it as a factor. Factors are useful for categorical variables and are discussed in more detail later.

The variables `Item`, `Response`, and `Answer` are identifiers:

- `Item` identifies the question answered by a participant.
- `Response` identifies the multiple-choice response selected by the participant.
- `Answer` identifies the correct response for that question.

These variables are currently encoded as integers. This will not cause a problem, but they could equally be represented as character values, such as `"Q1"` and `"Q2"`, without changing the meaning of the data.

Finally, `Correct` records whether `Response` matches `Answer`. Because it contains `TRUE` and `FALSE` values, it has been correctly imported as a logical variable.



Let’s work with a dataset that contains some common data-quality issues. The CSV file is stored online, so it can be imported with `read.csv()`:

```r
messyDat <- read.csv("https://raw.githubusercontent.com/PCHN63101/Importing-Wrangling-Summarising/main/messy.csv")
```

You can then inspect its structure using `str()`:

```r
str(messyDat)
```

```text
'data.frame':  9 obs. of  6 variables:
 $ P                    : chr  "P001" "P002" "P003" "P004" ...
 $ age                  : chr  "18" "19" "" "20" ...
 $ CongruentReactionTime: num  2.45 1.89 2.1 3 2.1 2.1 1.77 2.13 2.89
 $ Incongruent.RT       : num  3.25 2.99 NA 4.21 3.96 3.96 2.89 3.56 3.91
 $ Edu                  : chr  "School" "UG" "PG" "pg" ...
 $ Completed            : logi  TRUE TRUE TRUE TRUE TRUE TRUE ...
```

The dataset contains **nine rows** and **six columns**:

- Three character variables
- Two numeric variables
- One logical variable

Several issues are immediately visible:

- `age` is stored as a character variable, even though age should usually be numeric.
- `Edu` contains inconsistent category labels: both `"PG"` and `"pg"` appear.
- `Incongruent.RT` contains a missing value, represented by `NA`.

In the original CSV file, the incongruent reaction-time variable is named `Incongruent RT`. R replaces spaces in column names with dots by default, which is why it appears as `Incongruent.RT` after import.

Another useful function for exploring a dataset is `summary()`. It produces different output depending on the type of each variable:

```r
summary(messyDat)
```

```text
      P                 age            CongruentReactionTime Incongruent.RT      Edu            Completed      
 Length:9           Length:9           Min.   :1.77          Min.   :2.890   Length:9           Mode :logical  
 Class :character   Class :character   1st Qu.:2.10          1st Qu.:3.185   Class :character   FALSE:1        
 Mode  :character   Mode  :character   Median :2.10          Median :3.735   Mode  :character   TRUE :8        
                                       Mean   :2.27          Mean   :3.591                                     
                                       3rd Qu.:2.45          3rd Qu.:3.960                                     
                                       Max.   :3.00          Max.   :4.210                                     
                                                             NA's   :1
```

For numeric variables, R reports:

- Minimum
- First quartile
- Median
- Mean
- Third quartile
- Maximum
- Number of missing values, where applicable

If these data came from a Stroop experiment, the summary suggests that mean reaction time is slower in the incongruent condition than in the congruent condition.

In a Stroop task, participants name the font colour of a displayed word. Responses are typically faster when the word and font colour are congruent (for example, <span style="color: red;">red</span>) than when they are incongruent (for example, <span style="color: blue;">green</span>).

However, this initial summary should be interpreted cautiously. The dataset contains issues that should be cleaned before conducting further analysis.
